# VAYU Climate Digital Twin — Kaggle GPU Training (Indo-Gangetic Plain) v2

**Accelerator**: GPU T4 x2 (recommended)  
**Target**: R2_tmax >= 0.80, R2_rain >= 0.20 — Indo-Gangetic Plain, leakage-safe calendar split

## What changed in v2
- **Real static features**: DEM (Copernicus 90m) and land/sea mask (ESA WorldCover 2021)
  replace synthetic elevation/coastline geometry. Already baked into the sequence
  tensors — no extra files needed on Kaggle.
- **Leakage-safe calendar split**: train 2010–2021, validation 2022, held-out test
  2023–2025 (previously an 85/15 positional split with no test set).
- **Train-only normalization**: z-score statistics are fit on 2010–2021 only.
- **Held-out test evaluation runs automatically** after training and writes
  `test_report.json` with R²/RMSE/MAE + skill vs persistence/climatology.

## Region priority: temp_max (heat extremes, most populated region)
Unlike the other three regions, Indo-Gangetic Plain's dominant documented hazard is
heat extremes, not rainfall (per Int'l J. Climatology 2024, "Statistical downscaling
of maximum temperature... heat wave events... Indo-Gangetic Plain"). arXiv:2205.10972
("Global Extreme Heat Forecasting Using Neural Weather Models") found custom losses
tailored to temperature extremes give significant skill improvements over plain MSE.
temp_max weight raised 1.6 -> 2.0; rainfall weight stays at the global default (1.8).

## Required Dataset (Add Input -> Search by name)
Upload a **new version** of `shyam31415/vayu-indo-gangetic-plain-processed` containing:
- `train_sequences.pt`, `val_sequences.pt`, `test_sequences.pt` (17 features/node)
- `norm_params_2010-2025.nc`, `sequence_manifest.json`

No ERA5/NCEP wind data is available for this region — wind/humidity channels are
zero-filled. This is graceful degradation, consistent with the proven Western Ghats
baseline. Do not block training waiting for more downloads.

## Steps
1. Enable GPU: Settings -> Accelerator -> **GPU T4 x2**
2. Add the v2 dataset above via "Add Input"
3. Run all cells top to bottom

## v3 — why the previous run stalled, and what changed

The 2026-08-03 Western Ghats and North-East runs plateaued at **R2_rain ~ 0.001**
with **R2_tmax ~ 0.75** (barely above persistence). Four root causes were found
by measuring trivial predictors on the real validation data:

| Predictor (2022 val, WG, normalized space) | R2_rain | R2_tmax | R2_tmin |
|---|---|---|---|
| constant / dataset mean | -0.002 | -0.083 | -0.079 |
| persistence (the old skip connection) | **-0.303** | 0.722 | 0.721 |
| day-of-year climatology (train-years fit) | **+0.215** | 0.739 | 0.776 |
| 50/50 climatology + persistence | +0.153 | **+0.796** | **+0.804** |

A seasonal lookup table was beating the 6.6M-parameter model. Fixes:

1. **Rainfall loss: weighted MAE -> weighted MSE.** Absolute error is minimized
   by the conditional *median*, which for zero-inflated rain sits at the dry
   value. R2 scores the conditional *mean*. Measured: the old objective put
   98.6% of rainfall predictions at exactly 0.
2. **Removed the ReLU on rainfall.** Targets are per-cell z-scores in which
   45.4% of rainfall values are negative, so the clamp made the dry half of the
   distribution unrepresentable and pinned output at the mean (R2 = 0).
3. **Heads now blend persistence + day-of-year climatology** (learnable weights,
   climatology fitted on training years only, so no leakage). Rainfall starts on
   climatology because persistence scores -0.30 for it.
4. **Physics penalties off by default.** The conservation term was
   `|mean(pred) - mean(true)|`, which is minimized by predicting the mean, and
   the smoothness term suppressed the real terrain-driven temperature gradients
   that R2_tmax measures.
5. **8.5x more training data via `--all-windows`.** The pre-built bundles cap
   training at 512 windows; 2010-2021 offers ~4,350 at stride 1. Windows are now
   sliced lazily from `normalized_*.nc` (already inside this dataset), so no
   re-upload is needed. This notebook uses stride 3 to fit a Kaggle session.

**Starting point after these changes (before any training):**
R2_rain +0.215, R2_tmax +0.796, R2_tmin +0.804 — i.e. training now begins at
roughly the target instead of below it. Targets: R2_tmax >= 0.80, R2_rain >= 0.20.


In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected — switch accelerator to GPU in Settings!')
print('Python:', sys.version)

In [ ]:
!pip install -q torch-geometric==2.5.3 xarray netcdf4 typer scipy
print('Dependencies installed')

In [ ]:
import sys, os
from pathlib import Path

REGION = 'indo_gangetic_plain'
REPO_DIR = '/kaggle/working/isro'
PROCESSED_DIR = f'{REPO_DIR}/data/processed_{REGION}'
CHECKPOINT_DIR = f'{REPO_DIR}/checkpoints/{REGION}_main'

if os.path.exists(f'{REPO_DIR}/.git'):
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'rm -rf {REPO_DIR}')
    os.system(f'git clone https://github.com/Shyamistic/vayu.git {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working dir:', os.getcwd())

# NOTE: must run after clone/rm -rf above, since these dirs are nested inside
# REPO_DIR and would otherwise be wiped out by rm -rf.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

required = ['train_sequences.pt', 'val_sequences.pt', 'test_sequences.pt', 'sequence_manifest.json']
root = Path('/kaggle/input')
found = {name: next(iter(root.rglob(name)), None) for name in required}
missing = [k for k, v in found.items() if v is None]
if missing:
    raise RuntimeError('Missing dataset files: ' + ', '.join(missing) + ". Attach the v2 vayu-indo-gangetic-plain-processed dataset via 'Add Input'.")
parent_counts = {}
for p in found.values():
    parent_counts[str(p.parent)] = parent_counts.get(str(p.parent), 0) + 1
DATASET_DIR = max(parent_counts, key=parent_counts.get)
print('Dataset dir:', DATASET_DIR)

In [ ]:
# ── Stage bundle files (resilient to multi-folder Kaggle datasets) ──────────
# Files are located individually rather than from one DATASET_DIR, because a
# bundle is often split across sibling folders (…-1-001 / …-1-002). Copying from
# a single folder silently skipped normalized_*.nc / lsm.nc / test_sequences.pt.
import shutil
from pathlib import Path as _P

_ROOT = _P('/kaggle/input')

# Needed for lazy sliding windows (--all-windows) and for metrics.
REQUIRED_FILES = [
    'sequence_manifest.json',
    'norm_params_2010-2025.nc',
    'normalized_2010-2025.nc',
    'elevation.nc',
    'lsm.nc',
]
# train/val are only needed for the pre-built path and the smoke check;
# test_sequences enables held-out evaluation when not using --all-windows.
OPTIONAL_FILES = ['train_sequences.pt', 'val_sequences.pt', 'test_sequences.pt']


def _locate(name):
    """Find `name` anywhere under /kaggle/input, preferring this region's bundle."""
    matches = sorted(_ROOT.rglob(name))
    if not matches:
        return None
    preferred = [m for m in matches if f'kaggle_bundle_{REGION}' in str(m)]
    return (preferred or matches)[0]


_missing = []
for _f in REQUIRED_FILES + OPTIONAL_FILES:
    _src = _locate(_f)
    if _src is None:
        if _f in REQUIRED_FILES:
            _missing.append(_f)
        else:
            print(f'optional, not found: {_f}')
        continue
    shutil.copy(_src, PROCESSED_DIR)
    print(f'staged {_f:28s} <- {_src.parent}')

if _missing:
    raise RuntimeError(
        'Missing required files: ' + ', '.join(_missing) +
        '. Attach the complete v2 bundle (all sibling folders) via "Add Input".'
    )

os.system(f'ls -lah {PROCESSED_DIR}')


In [ ]:
import subprocess, sys, torch, json
PY = sys.executable

manifest = json.loads((_P(PROCESSED_DIR) / 'sequence_manifest.json').read_text())
print('Splits:', {k: v['saved_sequences'] for k, v in manifest['splits'].items()})
print('Feature count:', manifest['feature_count'])

_seqs = torch.load(f'{PROCESSED_DIR}/train_sequences.pt', map_location='cpu', weights_only=False)
_nf = _seqs[0][0].x.shape[-1]
_nn = _seqs[0][0].x.shape[0]
print(f'Sequence feature count: {_nf} (expected 17) | nodes: {_nn}')
if _nf != 17:
    raise RuntimeError(f'STALE SEQUENCES: found {_nf} features, expected 17.')
del _seqs
print('✓ Sequences verified')

_r = subprocess.run([PY, '-m', 'ai_engine.trainer',
    '--data-dir', PROCESSED_DIR,
    '--checkpoint-dir', f'{REPO_DIR}/checkpoints/{REGION}_smoke',
    '--epochs', '1', '--device', 'auto', '--smoke-only'],
    cwd=REPO_DIR, capture_output=True, text=True)
print(_r.stdout[-3000:] if _r.stdout else '')
if _r.returncode != 0:
    print('\n=== smoke STDERR ===')
    print(_r.stderr[-4000:] if _r.stderr else '(empty)')
    raise RuntimeError(f'Smoke check failed (exit {_r.returncode})')
print('\n✓ Smoke check PASSED')

In [ ]:
# Indo-Gangetic Plain has ~2205 nodes (largest of the 4 regions) — full v2 architecture
# still fits comfortably on T4; no lite/medium preset needed.
# --tmax-weight 2.0 biases the loss toward this region's heat-extreme priority (see
# markdown header for citations). Held-out test evaluation runs automatically at
# the end of training since test_sequences.pt is present, writing test_report.json.
import os, subprocess, sys
PY = sys.executable
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run([PY, '-m', 'ai_engine.trainer',
    '--data-dir',               PROCESSED_DIR,
    '--checkpoint-dir',         CHECKPOINT_DIR,
    '--epochs',                 '40',
    '--device',                 'auto',
    '--amp',
    '--batch-size',              '1',
    '--grad-accum-steps',       '8',
    '--cosine-lr',
    '--early-stopping-patience', '20',
    '--weight-decay',           '1e-4',
    '--gnn-dropout',            '0.12',
    '--lambda-conservation',    '0.0',
    '--lambda-smoothness',      '0.0',
    '--tmax-weight',            '2.0',
    '--norm-params-file',       f'{PROCESSED_DIR}/norm_params_2010-2025.nc',
    '--normalized-file',        f'{PROCESSED_DIR}/normalized_2010-2025.nc',
    '--elevation-file',         f'{PROCESSED_DIR}/elevation.nc',
    '--lsm-file',               f'{PROCESSED_DIR}/lsm.nc',
    '--all-windows',
    '--train-stride',           '4',
    '--eval-stride',            '3',
    '--run-baselines',
    '--require-benchmarks'],
    check=True, cwd=REPO_DIR)

os.system(f'ls -lah {CHECKPOINT_DIR}')

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

history_path = Path(CHECKPOINT_DIR) / 'training_history.json'
if not history_path.exists():
    print('No training_history.json yet — run the training cell first.')
else:
    history = json.loads(history_path.read_text())
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history['epochs'], history['train_loss'], label='Train Loss')
    axes[0].plot(history['epochs'], history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Indo-Gangetic Plain Training Loss')
    axes[0].legend(); axes[0].grid(True)
    axes[1].plot(history['epochs'], history['val_r2'], color='green', label='R2 Tmax')
    axes[1].axhline(0.80, color='red', linestyle='--', label='Target R2=0.80')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('R2'); axes[1].set_title('Validation R2')
    axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves_indo_gangetic_plain.png', dpi=150)
    plt.show()
    print('Best val_loss:', min(history['val_loss']))
    if history['benchmark_metrics']:
        last = history['benchmark_metrics'][-1]
        print(f"Latest validation R2_tmax={last.get('r2_tmax'):.3f} | R2_tmin={last.get('r2_tmin'):.3f} | R2_rain={last.get('r2_rain'):.3f}")

In [ ]:
# ── Held-out test set results (2023–2025, never seen during training/validation) ──
import json
from pathlib import Path

test_report_path = Path(CHECKPOINT_DIR) / 'test_report.json'
if test_report_path.exists():
    test_results = json.loads(test_report_path.read_text())
    for var, metrics in test_results.items():
        print(f"{var}: R2={metrics['r2']:.3f} | RMSE={metrics['rmse']:.3f} | MAE={metrics['mae']:.3f} | "
              f"skill_vs_persistence={metrics['skill_vs_persistence']:.3f} | skill_vs_climatology={metrics['skill_vs_climatology']:.3f}")
else:
    print('No test_report.json found — ensure test_sequences.pt was present before training.')

In [ ]:
import shutil
from pathlib import Path

best_ckpt = Path(CHECKPOINT_DIR) / 'vayu_best.pt'
if best_ckpt.exists():
    dst = f'/kaggle/working/vayu_best_{REGION}.pt'
    shutil.copy(best_ckpt, dst)
    size_mb = best_ckpt.stat().st_size / 1e6
    print(f'Checkpoint ready: {dst} ({size_mb:.1f} MB)')
else:
    print('vayu_best.pt not found — check training cell output for errors.')